# 03 PyTorch Training v1

Persistent PyTorch warm-up + fine-tune test run. This notebook uses the existing NB02 split manifests and `src_torch` helpers; TensorFlow notebooks are not modified.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ.setdefault('MPLCONFIGDIR', '/private/tmp')

print('PROJECT_ROOT:', PROJECT_ROOT)

PROJECT_ROOT: /Users/starsrain/2025_codeProject/GreenSpace_CNN


In [2]:
import torch
import torchvision
import torchgeo

from src_torch.config import TORCH_DATA_CONFIG, TORCH_MODEL_CONFIG, TORCH_TRAINING_CONFIG
from src_torch.training import run_persistent_warmup_finetune

print('torch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('torchgeo:', torchgeo.__version__)
print('TORCH_DATA_CONFIG:', TORCH_DATA_CONFIG)
print('TORCH_MODEL_CONFIG:', TORCH_MODEL_CONFIG)
print('TORCH_TRAINING_CONFIG:', TORCH_TRAINING_CONFIG)

torch: 2.10.0
torchvision: 0.25.0
torchgeo: 0.8.1
TORCH_DATA_CONFIG: {'img_size': (512, 512), 'batch_size': 4, 'num_workers': 0, 'pin_memory': False, 'image_transform': 'tf_parity', 'backbone_preprocess': None, 'use_oversampling': True, 'use_augmentation': True, 'oversampling_sanity_batches': 80}
TORCH_MODEL_CONFIG: {'backbone_priority': 'torchgeo', 'torchgeo_model_name': 'resnet50', 'torchgeo_weight': 'ResNet50_Weights.FMOW_RGB_GASSL', 'load_pretrained_weights': True, 'fallback_backbone': 'torchvision', 'num_binary': 7, 'num_shade': 2, 'score_output_range': (1.0, 5.0), 'veg_output_range': (1.0, 5.0)}
TORCH_TRAINING_CONFIG: {'seed': 37, 'test_run_mode': True, 'test_warmup_epochs': 1, 'test_finetune_epochs': 1, 'warmup_epochs': 5, 'finetune_epochs': 100, 'warmup_learning_rate': 0.001, 'finetune_learning_rate': 0.0001, 'fine_tune_backbone': True, 'device': 'auto', 'max_train_batches': None, 'max_val_batches': None}


## Run 1+1 Training Test

Expected behavior: one warm-up epoch with the TorchGeo backbone frozen, then one fine-tune epoch with the backbone trainable when `fine_tune_backbone=True`. Artifacts are saved under `models/runs/PyTorch_<timestamp>/`.

In [3]:
# First persistent pipeline test: cap batches so CPU-only environments finish quickly.
# Set both caps to None when you are ready for a full-split 1+1 run.
training_override = {
    'max_train_batches': 20,
    'max_val_batches': 20,
}

result = run_persistent_warmup_finetune(training_config=training_override)
result

RUN_TAG: PyTorch_20260531_202444
RUN_DIR: /Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260531_202444
device: mps
test_run_mode=True -> warmup=1, finetune=1
batch caps: train=20, val=20
oversampling plan: {'active': True, 'streams': [{'label': 'children_s_playground_p', 'target_rate': 0.2, 'pos_threshold': 0.5, 'row_count': 381, 'base_soft_mean': 0.1127757852407347, 'threshold_rate': 0.12422562764916857}, {'label': 'water_feature_p', 'target_rate': 0.25, 'pos_threshold': 0.5, 'row_count': 557, 'base_soft_mean': 0.18941419410933596, 'threshold_rate': 0.20052168242582327}], 'remainder_count': 2129, 'remainder_weight': 0.55, 'weights': [0.2, 0.25, 0.55]}
Warm-up trainable summary: {'total_params': 23530571, 'trainable_params': 22539, 'frozen_params': 23508032, 'trainable_groups': ['bin_head', 'shade_head', 'score_head_raw', 'veg_head_raw'], 'groups': {'backbone': {'total': 23508032, 'trainable': 0}, 'bin_head': {'total': 14343, 'trainable': 14343}, 'shade_head': {

warmup epoch 1: train_loss=3.3026 val_loss=3.0371


Fine-tune trainable summary: {'total_params': 23530571, 'trainable_params': 23530571, 'frozen_params': 0, 'trainable_groups': ['backbone', 'bin_head', 'shade_head', 'score_head_raw', 'veg_head_raw'], 'groups': {'backbone': {'total': 23508032, 'trainable': 23508032}, 'bin_head': {'total': 14343, 'trainable': 14343}, 'shade_head': {'total': 4098, 'trainable': 4098}, 'score_head_raw': {'total': 2049, 'trainable': 2049}, 'veg_head_raw': {'total': 2049, 'trainable': 2049}}}


finetune epoch 2: train_loss=3.0228 val_loss=2.9140


Saved final checkpoint: /Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260531_202444/final_PyTorch_20260531_202444.pt
Saved best MC-MAE checkpoint: True /Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260531_202444/best_mcmae_PyTorch_20260531_202444.pt
Saved best PR-AUC checkpoint: True /Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260531_202444/best_prauc_PyTorch_20260531_202444.pt
Saved config: /Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260531_202444/model_config_PyTorch_20260531_202444.json
Saved history: /Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260531_202444/training_history_PyTorch_20260531_202444.json
Saved curves: /Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260531_202444/training_curves.png


{'run_tag': 'PyTorch_20260531_202444',
 'run_dir': '/Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260531_202444',
 'final_model_path': '/Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260531_202444/final_PyTorch_20260531_202444.pt',
 'best_mc_mae_path': '/Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260531_202444/best_mcmae_PyTorch_20260531_202444.pt',
 'best_prauc_path': '/Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260531_202444/best_prauc_PyTorch_20260531_202444.pt',
 'model_config_path': '/Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260531_202444/model_config_PyTorch_20260531_202444.json',
 'training_history_path': '/Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260531_202444/training_history_PyTorch_20260531_202444.json',
 'training_curves_path': '/Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260531_202444/tra

In [4]:
run_dir = Path(result['run_dir'])
print('RUN_DIR:', run_dir)
for path in sorted(run_dir.iterdir()):
    print(path.name)

RUN_DIR: /Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260531_202444
best_mcmae_PyTorch_20260531_202444.pt
best_prauc_PyTorch_20260531_202444.pt
final_PyTorch_20260531_202444.pt
model_config_PyTorch_20260531_202444.json
training_curves.png
training_history_PyTorch_20260531_202444.json
